# ⚗️ Machine Learning for Chemistry: Predicting Molecular Properties
### AI Literacy Project Taster Day — Queen Mary University of London

---

In the previous tutorials you learned how to represent molecules as SMILES strings
and convert them into numerical fingerprints. Now we put everything together:
**can a computer learn to predict a molecule's quantum mechanical properties
directly from its structure — without running a single quantum calculation?**

That is exactly what we will do today. We will train machine learning models to
predict the **HOMO-LUMO gap** of small organic molecules — a quantum property
that determines how a molecule absorbs light and behaves as a semiconductor.

By the end of this notebook you will be able to:
- Load and explore the **QM9 dataset** — a benchmark dataset of 134,000 molecules with DFT-calculated properties
- Represent molecules as **Morgan fingerprints** suitable for ML
- Split data correctly into **train / validation / test** sets
- Train and compare **three regression models**: Linear Regression, Random Forest, and Kernel Ridge Regression
- Evaluate model performance using **R², MAE, and RMSE**
- Critically assess what these results mean for real-world applications

> 💡 **How to run a cell:** Click on it, then press **Shift + Enter**.  
> Run cells **in order from top to bottom**.

---

| Section | Content |
|---------|---------|
| 1 | The QM9 dataset — what is the HOMO-LUMO gap? |
| 2 | Molecular representation — Morgan fingerprints |
| 3 | Splitting the dataset |
| 4 | Training and evaluating three ML models |
| 5 | Comparing models and discussion |

---


## Section 0 — Installing Required Packages 🛠️

We need three packages:

| Package | What it does |
|---------|-------------|
| `deepchem` | Chemistry-aware ML toolkit; we use it for its fingerprint featuriser |
| `rdkit` | Draw molecules and manipulate SMILES |
| `fast_ml` | Convenient train/validation/test splitting in one step |

In [ ]:
# ── Install required packages (TensorFlow NOT needed for this tutorial) ────────
! pip install deepchem rdkit fast_ml -q
print("✅ Packages installed.")


---
## Section 1 — The QM9 Dataset 📊

### What is QM9?

**QM9** is one of the most important benchmark datasets in computational chemistry.
It contains **134,000 small organic molecules** (containing C, H, O, N, F atoms,
up to 9 heavy atoms) with 19 quantum-mechanical properties calculated using
Density Functional Theory (DFT).

Running a single DFT calculation takes minutes to hours on a computer cluster.
Training an ML model on QM9 takes minutes — and prediction for a new molecule
takes milliseconds. This is the promise of ML for chemistry.

### Our target: the HOMO-LUMO gap

The **gap** column contains the HOMO-LUMO energy gap (in Hartree, ≈27.2 eV).

| Term | Full name | What it means |
|------|-----------|--------------|
| HOMO | Highest Occupied Molecular Orbital | The "top floor" of electron energy levels |
| LUMO | Lowest Unoccupied Molecular Orbital | The first "empty floor" above |
| Gap  | HOMO-LUMO gap | The energy needed to excite an electron |

> 💬 **Why does the gap matter?**  
> - Small gap → molecule absorbs visible light → useful for solar cells, OLEDs  
> - Large gap → transparent, chemically stable → useful for insulators  
> Drug molecules and organic semiconductors are designed with specific gaps in mind.

---
### 1.1 — Load the dataset


In [ ]:
# import that pandas library
import pandas as pd

# load the dataframe as CSV from URL.
# If you upload the file to Colab, replace the URL with the file name
df = pd.read_csv("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv")

# Otherwise, load the dataframe as CSV from your current folder.
#df = pd.read_csv("./qm9.csv")

# look at the top 5 entries
df.head()

> 💬 The dataset has **134,885 molecules** and **21 columns** — SMILES plus
> 19 quantum properties plus a molecule ID. We will focus only on `smiles`
> and `gap` today.

---
### 1.2 — Visualise some molecules


In [ ]:
# import the Chem library for converting SMILES into RDKit molecules
from rdkit import Chem

# We will use MolToGridImage to visualize the 2D structure
from rdkit.Chem.Draw import MolsToGridImage

Let's look at a random sample of 20 molecules to get a feel for the dataset.
These are all small organic molecules — the kind found in pharmaceuticals and
organic electronics.


In [ ]:
# randomly select 20 entries from the dataframe
sample_df = df.sample(n=20)

# create a list of smiles
smiles_list = sample_df["smiles"].tolist()

# create the RDKit molecule objects with list comprehension
MolsToGridImage([Chem.MolFromSmiles(smile) for smile in smiles_list ])

---
### 1.3 — Explore the target property (HOMO-LUMO gap)

Before training any model, always visualise your target variable.
A well-distributed target is easier to learn.


First let's look at a small random sample of 20 molecules:


In [ ]:
# plotting the histogram from the sample dataframe
sample_df["gap"].plot(kind="hist")

> 💬 **Notice the gap** (missing values around 0.20–0.22 eV) — this small
> sample is not representative. An ML model trained only on this would perform
> poorly in that region. This is why we always use the **full dataset**.

Now let's look at the full QM9 distribution:


In [ ]:
# Looking at the whole QM9 dataset
df["gap"].plot(kind="hist")

> 💬 The full distribution is much smoother and covers the range evenly.
> This is a well-behaved dataset for regression — no extreme outliers and
> a roughly normal distribution.

---
### 1.4 — Create a working subset

The full QM9 has 134,000 molecules. For today's workshop we'll take a **random
20% sample** (~27,000 molecules) so training runs in reasonable time.


In [ ]:
# create the dataset with only smiles and gap and 20% dataset
dataset = df[["smiles","gap"]].sample(frac=0.2)

# look at the top 5 entries
dataset.head()

> 💬 We use `sample(frac=0.2)` to randomly select 20% of rows.
> `random_state` is not set here — run the cell twice and notice the index
> numbers change! For reproducibility in research you would set `random_state=42`.

---


## Section 2 — Molecular Representation: Morgan Fingerprints 🔬

As you learned in the fingerprints tutorial, ML models need **numbers** as
input — not SMILES strings or 2D drawings.

We use DeepChem's `CircularFingerprint` featuriser, which generates
**Morgan fingerprints** with:
- `radius=2` — look at each atom and its neighbours up to 2 bonds away
- `size=100` — encode into a 100-bit vector

> 💬 In research, 512 or 2048 bits are common — but 100 bits keeps training
> fast for this workshop. You can experiment with larger sizes later.


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="DEPRECATION WARNING: please use MorganGenerator")

In [ ]:

# --- Silenziare RDKit il prima possibile ---
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')   # oppure: RDLogger.logger().setLevel(RDLogger.ERROR)

# (Facoltativo) Silenziare eventuali Python warnings di DeepChem
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="deepchem")

# --- Ora importa DeepChem e usa MorganGenerator ---
import deepchem as dc

featurizer = dc.feat.CircularFingerprint(size=100, radius=2)

dataset["fp"] = dataset["smiles"].apply(lambda s: featurizer.featurize([s])[0])

dataset.head()

> 💬 **What just happened?**  
> `featurizer.featurize([s])[0]` converts one SMILES string into a 100-element
> numpy array of 0s and 1s.  
> We applied this to every row in the dataset using `apply()` — the same
> pattern you saw in the fingerprints tutorial.
>
> Each row in the `fp` column now contains a fingerprint vector like
> `[0, 1, 0, 0, 1, 1, 0, ...]` — ready to be fed into a ML model.

---


## Section 3 — Splitting the Dataset ✂️

### Why three splits?

| Split | Purpose | Fraction |
|-------|---------|---------|
| **Training set** | Model learns from this | 80% |
| **Validation set** | Tune hyperparameters; catch overfitting | 10% |
| **Test set** | Final, unbiased evaluation — seen only once | 10% |

> 💬 **What is overfitting?**  
> A model that memorises the training data but fails on new data.  
> Like a student who memorises past exam questions but can't answer new ones.  
> The validation set helps us detect this — if validation score drops while
> training score stays high, the model is overfitting.


In [ ]:
# install Fast-ML
! pip install fast_ml

`fast_ml`'s `train_valid_test_split` creates all three splits in one step —
more convenient than calling scikit-learn's `train_test_split` twice.


In [ ]:
# import the function to split into train-valid-test
from fast_ml.model_development import train_valid_test_split

# we will split the dataset as train-valid-test = 0.8:0.1:0.1
X_train, y_train, X_valid, y_valid, \
X_test, y_test = train_valid_test_split(dataset[["fp","gap"]], target = "gap", train_size=0.8,
                                        valid_size=0.1, test_size=0.1)

Let's inspect what these splits look like:


In [ ]:
# look at the dataset
X_train

In [ ]:
# look at the dataset
y_train

> 💬 `X_train` contains the `fp` (fingerprint) column — this is our input to
> the model. `y_train` contains the `gap` values — this is what we want to
> predict. Notice each fingerprint is a numpy array inside the DataFrame.

---


## Section 4 — Training Machine Learning Models 🧠

We will train **three models** and compare their performance:

| Model | Key idea | Strength |
|-------|---------|----------|
| **Linear Regression** | Fits a straight line through the data | Fast, interpretable, good baseline |
| **Random Forest** | Ensemble of many decision trees | Handles non-linearity, robust |
| **Kernel Ridge Regression** | Projects data into a higher-dimensional space | Captures complex patterns |

For each model we follow the same three steps:
1. **Train** on the training set
2. **Validate** on the validation set (tune if needed)
3. **Test** on the test set (report final performance)


---
### 4.1 — Linear Regression

Linear regression fits the equation:

$$\text{gap} = w_1 \times \text{bit}_1 + w_2 \times \text{bit}_2 + \ldots + w_{100} \times \text{bit}_{100} + b$$

It finds the weights $w_1 \ldots w_{100}$ that minimise the difference between
predicted and actual gap values. This is the same $y = mx + c$ you know from
A-level Maths, extended to 100 dimensions.

#### Training


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Convert fingerprints into a proper 2D NumPy array
X_train_matrix = np.stack(X_train["fp"].values)  # shape: (n_samples, n_features)
y_train_array = y_train.values                   # shape: (n_samples,)

# Create and fit the model
lr = LinearRegression()
lr.fit(X_train_matrix, y_train_array)

> 💬 `np.stack()` converts the list of fingerprint arrays in the `fp` column
> into a single 2D matrix of shape `(n_samples, 100)` — the format scikit-learn
> expects.

#### Validation


In [ ]:
# Convert fingerprints to a proper 2D NumPy array
X_valid_matrix = np.stack(X_valid["fp"].values)
y_valid_array = y_valid.values

# Compute R² score
r2 = lr.score(X_valid_matrix, y_valid_array)
print(f"Validation R²: {r2:.3f}")

> 💬 **R² (R-squared):** measures how much of the variation in gap values the
> model explains. R² = 1 means perfect prediction; R² = 0 means the model is
> no better than always predicting the mean.

#### Test


In [ ]:
# Convert fingerprints to 2D NumPy array
X_test_matrix = np.stack(X_test["fp"].values)
y_test_array = y_test.values

# Predict on test set
y_test_pred = lr.predict(X_test_matrix)

# Evaluate performance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np  # Make sure numpy is imported

r2 = r2_score(y_test_array, y_test_pred)
mae = mean_absolute_error(y_test_array, y_test_pred)
mse = mean_squared_error(y_test_array, y_test_pred)  # Calculate MSE first
rmse = np.sqrt(mse)  # Then take the square root manually

print(f"Test R²: {r2:.3f}, MAE: {mae:.3f}, RMSE: {rmse:.3f}")

> 💬 **MAE** (Mean Absolute Error) — average error in the same units as gap (Hartree).  
> **RMSE** (Root Mean Squared Error) — penalises large errors more heavily than MAE.

---
### 4.2 — Random Forest 🌳

A **Random Forest** builds hundreds of decision trees, each trained on a random
subset of the data and features. The final prediction is the average across
all trees. This makes it:
- More powerful than linear regression for non-linear relationships
- Robust to noise and outliers
- Less prone to overfitting than a single decision tree

#### Training


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Convert fingerprints into a 2D NumPy matrix
X_train_matrix = np.stack(X_train["fp"].values)
y_train_array  = y_train.values

# Train the Random Forest
# n_estimators = number of trees; more trees = better but slower
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_matrix, y_train_array)

print("✅ Random Forest trained.")


> 💬 `n_estimators=100` means we build 100 decision trees.  
> `n_jobs=-1` tells scikit-learn to use all available CPU cores — this speeds
> up training significantly.

#### Validation


In [ ]:
X_valid_matrix = np.stack(X_valid["fp"].values)
y_valid_array  = y_valid.values

r2_rf_valid = rf.score(X_valid_matrix, y_valid_array)
print(f"Random Forest — Validation R²: {r2_rf_valid:.3f}")


> 💬 Compare this R² to the Linear Regression validation score.
> Which model fits the validation data better?

#### Test


In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

X_test_matrix = np.stack(X_test["fp"].values)
y_test_array  = y_test.values

y_test_pred_rf = rf.predict(X_test_matrix)

r2_rf   = r2_score(y_test_array, y_test_pred_rf)
mae_rf  = mean_absolute_error(y_test_array, y_test_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_array, y_test_pred_rf))

print(f"Random Forest — Test R²: {r2_rf:.3f}, MAE: {mae_rf:.4f}, RMSE: {rmse_rf:.4f}")


---
### 4.3 — Kernel Ridge Regression (KRR)

**Kernel Ridge Regression** uses a mathematical trick called the **kernel trick**:
it implicitly maps the fingerprint vectors into a much higher-dimensional space
where a linear model can capture non-linear patterns in the original space.

The RBF (Radial Basis Function) kernel measures similarity between two
fingerprints based on their Euclidean distance:

$$K(x, x') = \exp\left(-\gamma \|x - x'\|^2\right)$$

Two hyperparameters control the model:
- `alpha` — regularisation strength (larger = simpler model, less overfitting)
- `gamma` — width of the RBF kernel (larger = more local, more complex)

#### Training


In [ ]:
import numpy as np
from sklearn.kernel_ridge import KernelRidge

# Convert fingerprints into a proper 2D NumPy array
X_train_matrix = np.stack(X_train["fp"].values)  # shape: (n_samples, n_features)
y_train_array = y_train.values                   # shape: (n_samples,)

# Create and fit the Kernel Ridge Regression model
krr = KernelRidge(kernel='rbf', alpha=1.0, gamma=0.1)  # You can tune alpha and gamma
krr.fit(X_train_matrix, y_train_array)

> 💬 KRR can be slower to train than linear regression or random forest on
> large datasets — because it needs to compute the kernel matrix between all
> training pairs. That is why we work with a 20% subset of QM9.

#### Validation


In [ ]:
# Convert fingerprints to a proper 2D NumPy array
X_valid_matrix = np.stack(X_valid["fp"].values)
y_valid_array = y_valid.values

# Compute R² score
r2 = krr.score(X_valid_matrix, y_valid_array)
print(f"Validation R²: {r2:.3f}")

#### Test


In [ ]:
# Convert fingerprints to 2D NumPy array
X_test_matrix = np.stack(X_test["fp"].values)
y_test_array = y_test.values

# Predict on test set
y_test_pred = krr.predict(X_test_matrix)

# Evaluate performance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np  # Make sure numpy is imported

r2 = r2_score(y_test_array, y_test_pred)
mae = mean_absolute_error(y_test_array, y_test_pred)
mse = mean_squared_error(y_test_array, y_test_pred)  # Calculate MSE first
rmse = np.sqrt(mse)  # Then take the square root manually

print(f"Test R²: {r2:.3f}, MAE: {mae:.3f}, RMSE: {rmse:.3f}")

---
## Section 5 — Model Comparison & Discussion 📊

### 5.1 — Side-by-side results table

Run the cell below to see all three models compared in a single table.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ── Collect predictions from all three models ──────────────────────────────────
X_test_matrix = np.stack(X_test["fp"].values)
y_test_array  = y_test.values

models = {
    "Linear Regression" : lr,
    "Random Forest"     : rf,
    "Kernel Ridge (KRR)": krr,
}

rows = []
for name, model in models.items():
    y_pred = model.predict(X_test_matrix)
    r2     = r2_score(y_test_array, y_pred)
    mae    = mean_absolute_error(y_test_array, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test_array, y_pred))
    rows.append({"Model": name, "R²": round(r2, 3),
                 "MAE (Ha)": round(mae, 4), "RMSE (Ha)": round(rmse, 4)})

results_df = pd.DataFrame(rows).set_index("Model")
print("Model Comparison — Test Set Performance\n")
print(results_df.to_string())


---
### 5.2 — Actual vs Predicted plots


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
model_names = ["Linear Regression", "Random Forest", "Kernel Ridge (KRR)"]
model_list  = [lr, rf, krr]
colors      = ["#4C72B0", "#55A868", "#C44E52"]

for ax, name, model, color in zip(axes, model_names, model_list, colors):
    y_pred = model.predict(X_test_matrix)
    ax.scatter(y_test_array, y_pred, alpha=0.2, s=8, color=color)
    lo = min(y_test_array.min(), y_pred.min())
    hi = max(y_test_array.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.5, label="Perfect prediction")
    r2 = r2_score(y_test_array, y_pred)
    ax.set_title(f"{name}\nR² = {r2:.3f}", fontsize=10)
    ax.set_xlabel("Actual HOMO-LUMO gap (Ha)")
    ax.set_ylabel("Predicted gap (Ha)")
    ax.legend(fontsize=8)

plt.suptitle("Actual vs Predicted HOMO-LUMO Gap — Test Set", fontsize=13)
plt.tight_layout()
plt.show()


> 💬 **How to read these plots:**  
> If every prediction were perfect, all points would sit exactly on the dashed
> black line. Scatter around the line = prediction error.  
> Which model has the tightest scatter?

---
### 5.3 — Discussion Questions 💬

1. **Model ranking:** Which model performed best by R²? Does this match your
   intuition — why might a Random Forest outperform Linear Regression on
   molecular property prediction?

2. **What R² means in practice:** If your best model achieves R² = 0.7, that
   means it explains 70% of the variation in gap values. Is this good enough
   to be useful in drug discovery? What would you need R² to be?

3. **The fingerprint bottleneck:** Our fingerprints encode *structural* information
   (which fragments are present) but not 3D information (bond angles, distances).
   The HOMO-LUMO gap depends heavily on 3D electronic structure. How might this
   limit our model?

4. **100-bit vs 2048-bit fingerprints:** We used `size=100` for speed.
   Try changing it to `size=512` or `size=2048` in Section 2 and re-running the
   notebook. Does model performance improve?

5. **Overfitting check:** Compare training R² and test R² for each model.
   Which model shows the largest gap between training and test performance?
   What does this tell you?

---
### What comes next in ML for Chemistry?

| Topic | Description |
|-------|-------------|
| **Better representations** | 3D descriptors, Coulomb matrices, graph neural networks |
| **Transfer learning** | Pre-train on large datasets (like QM9), fine-tune on small experimental datasets |
| **Active learning** | The model tells you which experiments to run next to improve itself |
| **Generative models** | Use ML to *design* new molecules with target properties, not just predict |

---
### Further Reading

- DeepChem documentation: https://deepchem.io/
- QM9 dataset paper: Ramakrishnan et al., *Sci. Data*, 2014, **1**, 140022
- MoleculeNet benchmark: Wu et al., *Chem. Sci.*, 2018, **9**, 513–530
- Review of ML for quantum chemistry: von Lilienfeld, *Angew. Chem.*, 2018, **57**, 4164

---
*Notebook developed for the QMUL AI Literacy Project Taster Day · Department of Chemistry*
